In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.silver_customers"
    )
)

In [0]:
silver_customers.printSchema()

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.silver_loads"
    )
)

In [0]:
silver_loads.printSchema()

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.silver_delivery_events"
    )
)

In [0]:
silver_delivery_events.printSchema()

In [0]:
from pyspark.sql import functions as F

silver_customers = spark.table(
    "workspace.transportation_analytics.silver_customers"
)

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

silver_delivery_events = spark.table(
    "workspace.transportation_analytics.silver_delivery_events"
)

print("Silver customer tables loaded successfully")

In [0]:
gold_customer_analysis = (
    silver_loads
    .groupBy("customer_id")
    .agg(
        F.countDistinct("load_id").alias("total_loads"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("fuel_surcharge").alias("total_fuel_surcharge"),
        F.sum("accessorial_charges").alias("total_accessorial_charges")
    )
    .join(
        silver_customers.select(
            "customer_id"
        ),
        on="customer_id",
        how="left"
    )
)

display(gold_customer_analysis)

In [0]:
customer_service = (
    silver_delivery_events
    .join(
        silver_loads.select(
            "load_id",
            "customer_id"
        ),
        on="load_id",
        how="left"
    )
    .groupBy("customer_id")
    .agg(
        F.countDistinct("load_id").alias("total_deliveries"),
        F.sum(
            F.when(F.col("on_time_flag") == True, 1).otherwise(0)
        ).alias("on_time_deliveries"),
        F.avg(
            F.when(F.col("on_time_flag") == True, 1.0).otherwise(0.0)
        ).alias("on_time_delivery_rate"),
        F.avg("detention_minutes").alias("avg_detention_minutes")
    )
)

display(customer_service)

In [0]:
gold_customer_analysis = (
    gold_customer_analysis
    .join(
        customer_service,
        on="customer_id",
        how="left"
    )
)

display(gold_customer_analysis)

In [0]:
gold_customer_analysis.limit(0).write \
    .format("delta") \
    .saveAsTable(
        "workspace.transportation_analytics.gold_customer_analysis"
    )

print("Empty Gold Customer Analysis table created successfully")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_customer_analysis"
)

target.alias("t").merge(
    gold_customer_analysis.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Customer Analysis table updated using MERGE")

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_customer_analysis"
    )
)

In [0]:
customer_delivery_status = (
    silver_delivery_events
    .join(
        silver_loads.select(
            "load_id",
            "customer_id"
        ),
        on="load_id",
        how="left"
    )
    .groupBy(
        "customer_id",
        "load_id"
    )
    .agg(
        F.max(
            F.when(F.col("on_time_flag") == True, 1).otherwise(0)
        ).alias("load_on_time"),
        F.avg("detention_minutes").alias("load_detention_minutes")
    )
)

customer_service = (
    customer_delivery_status
    .groupBy("customer_id")
    .agg(
        F.countDistinct("load_id").alias("total_deliveries"),
        F.sum("load_on_time").alias("on_time_deliveries"),
        F.avg("load_on_time").alias("on_time_delivery_rate"),
        F.avg("load_detention_minutes").alias("avg_detention_minutes")
    )
)

display(customer_service)

In [0]:
gold_customer_analysis = (
    gold_customer_analysis
    .drop(
        "total_deliveries",
        "on_time_deliveries",
        "on_time_delivery_rate",
        "avg_detention_minutes"
    )
    .join(
        customer_service,
        on="customer_id",
        how="left"
    )
)

display(gold_customer_analysis)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_customer_analysis"
)

target.alias("t").merge(
    gold_customer_analysis.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Customer Analysis table updated using MERGE")

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_customer_analysis"
    )
)

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_customer_analysis"
    )
)

In [0]:
display(
    gold_customer_analysis
    .select(
        "customer_id",
        "total_revenue"
    )
    .orderBy(F.desc("total_revenue"))
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_customer_analysis
    .select(
        "customer_id",
        "on_time_delivery_rate"
    )
    .orderBy(F.desc("on_time_delivery_rate"))
)

Databricks visualization. Run in Databricks to view.